# Final Project

This notebook is aligned with the current script responsibilities:
- `Final_Project.py`: generates `outputs/submission.csv`.
- `Final_Project_Eval.py`: runs offline evaluation and tuning.
- `experiments/`: tracks each run and official Kaggle scores.


## 1. 导入依赖与环境检查

In [ ]:
from pathlib import Path

from Final_Project import (
    BASE_PATH,
    OUTPUT_DIR,
    print_environment_info,
    print_data_file_status,
    load_tables,
    summarize_inputs,
    prepare_transactions,
    prepare_article_department,
    prepare_customer_age_bin,
    load_ranker_from_cache,
    describe_ranker,
    generate_submission,
    run_pipeline,
)

print_environment_info()
print_data_file_status(BASE_PATH)


## 2. 查看输入数据概览

In [ ]:
tables = load_tables(BASE_PATH)
summary = summarize_inputs(tables)
summary

## 3. 生成层：仅产出 submission.csv

这一步是“手动分步版”，和 `run_pipeline` 执行逻辑一致。

In [ ]:
transactions = prepare_transactions(tables["transactions"])
article_department = prepare_article_department(tables["articles"])
customer_age_bin = prepare_customer_age_bin(tables["customers"])

ranker_config = load_ranker_from_cache(output_dir=OUTPUT_DIR)
print("Selected ranker:", describe_ranker(ranker_config))

submission = generate_submission(
    tables=tables,
    transactions=transactions,
    article_department=article_department,
    customer_age_bin=customer_age_bin,
    ranker_config=ranker_config,
    output_dir=OUTPUT_DIR,
)

submission.head(5)


## 4. 检查输出路径

In [ ]:
output_path = Path(OUTPUT_DIR) / "submission.csv"
print(output_path)
print("exists:", output_path.exists())

## 5. 生成层：一键运行版

如需要一条命令跑完整生成流程，可使用下面入口。

In [ ]:
state = run_pipeline(BASE_PATH)
state["submission"].head(5)

## 6. Evaluation Layer (Optional, Run Separately)

Evaluation and tuning are separated into `Final_Project_Eval.py` to keep generation fast.

Run in a separate session when needed:
```bash
python Final_Project_Eval.py
```

After evaluation, sync experiment records:
- `experiments/tuning_runs/<run_id>/`
- `experiments/runs_index.csv`
- after manual Kaggle submit, update `experiments/kaggle_scores.csv`


In [ ]:
from Final_Project_Eval import run_eval_pipeline

# 可选：这一步计算耗时较长，默认注释
# eval_state = run_eval_pipeline(BASE_PATH)
# eval_state["fold_metrics"]